# Notebook 03 — Condition B: XGBoost with Missingness-Aware Preprocessing

## Requires
- `data/raw/` (training_setA and training_setB)

## Produces
- `results/models/xgb_condition_B.pkl`
- Row appended to `results/experiment_log.csv`

## Run Order
Run **AFTER** notebook 01 (or standalone — loads raw data directly).

## Feature engineering note
`add_lag_features()` is called AFTER the patient-level split and BEFORE imputation.
Strategy B then adds 36 missingness indicator columns on top.
Final feature count: 40 original + 48 lag + 36 missingness indicators = 124 features.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from sklearn.metrics import average_precision_score

from src.config import (
    RANDOM_SEED, DATA_DIR, MODELS_DIR, METRICS_DIR, EXPERIMENT_LOG,
    ALL_FEATURES, VITAL_SIGNS, OUTLIER_BOUNDS, LABEL_SHIFT_HOURS, SPLIT_RATIOS,
)
from src.data_loader import load_all_patients
from src.preprocessing import engineer_labels, clip_outliers, apply_strategy_B
from src.features import add_lag_features
from src.utils import set_all_seeds, create_patient_splits
from src.evaluate import select_threshold, compute_all_metrics, log_results

set_all_seeds(RANDOM_SEED)
print('Imports OK')

## 1. Load Data and Create Splits

In [ ]:
df = load_all_patients(DATA_DIR)
df, _ = engineer_labels(df)
df = clip_outliers(df)

patient_train, patient_val, patient_test = create_patient_splits(df)

train_df = df[df['patient_id'].isin(patient_train)].copy()
val_df   = df[df['patient_id'].isin(patient_val)].copy()
test_df  = df[df['patient_id'].isin(patient_test)].copy()

print(f'Train: {len(patient_train):,} | Val: {len(patient_val):,} | Test: {len(patient_test):,}')

## 2. Lag Feature Engineering

In [ ]:
# Add 48 temporal lag features — same as Condition A for fair comparison
# Must be called AFTER split and BEFORE imputation
train_df = add_lag_features(train_df)
val_df   = add_lag_features(val_df)
test_df  = add_lag_features(test_df)

print(f'Train columns after lag features: {len(train_df.columns)}')

## 3. Strategy B Preprocessing

In [ ]:
# Strategy B: forward-fill + missingness indicators + StandardScaler
# apply_strategy_B uses ALL_FEATURES for indicators but we need lag features too.
# We handle imputation manually to include ALL columns (original + lag + indicators).
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from src.utils import validate_no_nans

meta_cols = {'patient_id', 'hospital_id', 'SepsisLabel', 'EarlyLabel'}

def add_indicators_and_ffill(df):
    df = df.copy()
    feat_cols = [c for c in df.columns if c not in meta_cols]
    for feat in feat_cols:
        if df[feat].isna().any():
            df[f'{feat}_missing'] = df[feat].isna().astype('int8')
    df[feat_cols] = df.groupby('patient_id')[feat_cols].ffill()
    return df

train_proc = add_indicators_and_ffill(train_df)
val_proc   = add_indicators_and_ffill(val_df)
test_proc  = add_indicators_and_ffill(test_df)

feature_cols_B = [c for c in train_proc.columns if c not in meta_cols]
for proc_df in (val_proc, test_proc):
    for col in feature_cols_B:
        if col not in proc_df.columns:
            proc_df[col] = 0

X_train_raw = train_proc[feature_cols_B].values
X_val_raw   = val_proc[feature_cols_B].values
X_test_raw  = test_proc[feature_cols_B].values

y_train = train_df['EarlyLabel'].values
y_val   = val_df['EarlyLabel'].values
y_test  = test_df['EarlyLabel'].values

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_val_imp   = imputer.transform(X_val_raw)
X_test_imp  = imputer.transform(X_test_raw)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_imp)
X_val   = scaler.transform(X_val_imp)
X_test  = scaler.transform(X_test_imp)

print('NaN validation:')
for arr, name in [(X_train,'train_B'), (X_val,'val_B'), (X_test,'test_B')]:
    validate_no_nans(arr, name, feature_cols_B)

print(f'Features: {X_train.shape[1]} | Train rows: {X_train.shape[0]:,} | mean ≈ {X_train.mean():.4f}')

os.makedirs(MODELS_DIR, exist_ok=True)
joblib.dump(imputer,        f'{MODELS_DIR}strategy_B_lag_imputer.pkl')
joblib.dump(scaler,         f'{MODELS_DIR}strategy_B_lag_scaler.pkl')
joblib.dump(feature_cols_B, f'{MODELS_DIR}strategy_B_lag_feature_names.pkl')

## 4. XGBoost Hyperparameter Grid Search

In [ ]:
pos_count        = int(y_train.sum())
neg_count        = int(len(y_train) - pos_count)
scale_pos_weight = neg_count / pos_count
print(f'scale_pos_weight = {scale_pos_weight:.1f}')

param_grid = [
    {'max_depth': d, 'learning_rate': lr}
    for d  in [3, 4, 6]
    for lr in [0.05, 0.1]
]

best_val_auprc = 0.0
best_params    = None
best_model     = None

for params in param_grid:
    model = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=params['max_depth'],
        learning_rate=params['learning_rate'],
        scale_pos_weight=scale_pos_weight,
        eval_metric='aucpr',
        early_stopping_rounds=20,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    val_probs = model.predict_proba(X_val)[:, 1]
    val_auprc = average_precision_score(y_val, val_probs)
    print(f"depth={params['max_depth']} lr={params['learning_rate']} "
          f"val AUPRC={val_auprc:.4f}")
    if val_auprc > best_val_auprc:
        best_val_auprc = val_auprc
        best_params    = params
        best_model     = model

print(f'\nBest: {best_params}  val AUPRC: {best_val_auprc:.4f}')

## 5. Evaluate Best Model

In [ ]:
val_probs  = best_model.predict_proba(X_val)[:, 1]
test_probs = best_model.predict_proba(X_test)[:, 1]

threshold    = select_threshold(y_val, val_probs)
val_metrics  = compute_all_metrics(y_val,  val_probs,  threshold)
test_metrics = compute_all_metrics(y_test, test_probs, threshold)

print('\n=== CONDITION B RESULTS ===')
print(f'Test AUC-ROC : {test_metrics["auc_roc"]:.4f}')
print(f'Test AUPRC   : {test_metrics["auprc"]:.4f}')
print(f'Test F1      : {test_metrics["f1"]:.4f}')

log_results(
    condition='B',
    model='XGBoost',
    strategy='Strategy_B',
    val_metrics=val_metrics,
    test_metrics=test_metrics,
    hyperparams=best_params,
)

## 6. Save Model

In [ ]:
joblib.dump(best_model, f'{MODELS_DIR}xgb_condition_B.pkl')
print(f'Model saved → {MODELS_DIR}xgb_condition_B.pkl')
print(f'Features used: {best_model.n_features_in_}')